# TUM Dataset

In [ ]:
%load_ext autoreload
%autoreload 2
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)

import torch, random, os, cv2
import numpy as np

seed = 1

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(seed)

import seaborn as sns
sns.set_theme(style="whitegrid")

import matplotlib.pyplot as plt

In [ ]:
from headset_localization import *

dataset_location_fr2_desk = "../tum_datasets/rgbd_dataset_freiburg2_desk"

tum_robot_env, tum_headset_data = scanned_3d_environment_and_headset_recording_from_tum(
        folder=dataset_location_fr2_desk,
        rgb_camera_name="freiburg2",
        time_tolerance= 0.03,
        n_robot_images= 20,
        xyz_image_generation_config=XYZImageGenerationConfig(iforest_contamination = 0.4, use_depth_images_if_provided=False),
        xyz_image_alginment_config=ICPAlignmentConfig(do_alginment=True),
        intervall=(0.0, 0.1)
)

### 3d visualisation

In [ ]:
vis_robot_env, vis_headset, vis_both = False, False, True

if vis_robot_env:
    tum_robot_env.visualize_3d_data()
if vis_headset:
    tum_headset_data.visualize_3d_data()
if vis_both:
    visualize_robot_camera_environment_combo(robot_env=tum_robot_env, headset_data=tum_headset_data, vis_headset_camera_wireframes = False)

In [ ]:
default_point_predictor = PnPLocalizer(
        cam2_intrinsic_mtx=tum_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=tum_robot_env.robot_bgr_images,
        cam1_xyz_images=tum_robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndMatchLoMa(loma_variant='LoMaB128')
        )
)

PredictionOnDataset(
    predictor = default_point_predictor,
    headset_data = tum_headset_data, 
    gripping_error=FastGrippingError(points=tum_robot_env.robot_xyz_images, intrinsics=tum_headset_data.intrinsic_cam_mtx, visualize=False)
).print_summary()

In [ ]:
yolo = YOLOv26Segmenter("yoloe-26l-seg.pt", prompts=["monitor", "keyboard", "mouse", "teddy", "tape", "book", "cup", "telephone", "can", "office appliance"])

pnp_loma_localizer_f = GradableLocalizer(
    creator=PnPLocalizer.get_creation_function(
        cam2_intrinsic_mtx=tum_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndMatchLoMa('LoMaB128'),
            rotation_augmentations=[Augmentation],
            ransac_config=pose_estimation_ransaac_config_10ms,
            display_matching=False,
        )
    ),
    name="PnP-LoMa"
)

pnp_lg_localizer_f = GradableLocalizer(
    creator=PnPLocalizer.get_creation_function(
        cam2_intrinsic_mtx=tum_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Augmentation],
            ransac_config=pose_estimation_ransaac_config_10ms,
        )
    ),
    name="PnP-LightGlue"
)

pnpl_localizer_f = GradableLocalizer(
    creator=PnPLLocalizer.get_creation_function(
        cam2_intrinsic_mtx=tum_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations = [Augmentation],
            extract_and_match=ExtractAndMatchLoMa(loma_variant="LoMaB128"),
            ransac_config=pose_estimation_ransaac_config_less_precise,
        ),
        pnpl_optimisation_conf=PnPLOptimizerConfig(
            line_relevance=0.2
        ),
        line_matching_config = LineMatchingConfig(
            min_number_supporting_points = 2,
            better_factor=1.5
        ),
        debug_visualize_pnpl = False,
        debug_visualize_3d = False,
        debug_visualize_matching = False
    ),
    name= "PnP+L"
)

ellipse_localizer_f = GradableLocalizer(
    creator=EllipsoidLocalizer.get_creation_function(
        cam2_intrinsic_mtx=tum_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndLightGlue(),
            rotation_augmentations=[Augmentation],
            ransac_config=pose_estimation_ransaac_config_less_precise,
            display_matching=False
        ),
        pne_optimizer=PyposePNEOptimizer(),
        cam1_segmenter=yolo,
        cam2_segmenter=yolo,
        matching_config=GaussianMatchingConfig(dummy_value=0.001),
        ellipsoid_matching_config = PointCloudMatchingConfig(min_cluster_size=2),
        ellipsoid_fitter=LeastShellDistanceEllipsoidFitter(contamination=0.2, size_penalty=0.98),
        visualize_environment_generation = True,
        visualize_segmentation_masks=False,
        visualize_pne_optimisation=False
    ),
    name="Ellipse"
)

grader = NPredictors1DatasetGrader(
    gradable_pose_predictors= [pnp_lg_localizer_f, pnp_loma_localizer_f, ellipse_localizer_f, pnpl_localizer_f],
    compute_gripping_error=True,
    headset_data = tum_headset_data,
    robot_env = tum_robot_env,
)

In [ ]:

visualize_trajectories_3d = True
if visualize_trajectories_3d:
    grader.visualize_predictions_3d()

grader.print_summary()

fig1, ax1 = plt.subplots(1, 1, figsize = (12, 5))
grader.plot_time_series_error(ax1, TimeSeriesErrorType.ABS_TRANSLATIONAL)

fig2, ax2 = plt.subplots(1, 1, figsize = (12, 5))
grader.plot_time_series_error(ax2, TimeSeriesErrorType.ABS_ROTATIONAL)

fig1, axes = plt.subplots(1, 2, figsize = (12, 2.5))
grader.plot_error_vs_error(axes[0], SingleValueErrorType.ATE_RMSE_TRANSLATIONAL, SingleValueErrorType.ATE_RMSE_ROTATIONAL, adjust_texts=False, plot_legend=False)
grader.plot_error_vs_error(axes[1], SingleValueErrorType.MED_TRANSLATIONAL, SingleValueErrorType.MED_ROTATIONAL, adjust_texts=False, plot_legend=False)

fig1, ax = plt.subplots(1, 1, figsize = (12, 3.7))
grader.plot_prediction_times(ax=ax, rotate_x_labels=None)


print("\n\n")
grader.print_translational_error_under_limits(limits_m=[0.5, 1, 2, 3])

plt.show()